# 03rl · FunnyBirds MCBM (relabeled) — the causal test of the label critique

**Question:** is concept–class backwash *caused* by the species-level label
standardization, or intrinsic to the bottleneck?

- **Standard MCBM** (notebook 03) is trained on **species-level** labels — every image
  of a species shares one concept vector (concept = f(species)).
- **Relabeled MCBM** (this notebook) is trained on **image-level** labels: a concept is
  set to **0 when the part covers <5% of its species-median pixel count** — so an
  occluded part is labelled absent. This **breaks the concept↔species correlation.**

**Prediction.** If the species-level labels *caused* backwash, the relabeled model's tail
should become **grounded** (z-ordering ↑, deletion retention ↓). If backwash persists, it
is intrinsic. This is the FunnyBirds twin of the CUB majority-vote result (notebook 04),
with an actual retrained model.

*Ref: `fb_mcbm_rl_renderer_swap.ipynb`. Pipeline to populate: relabel → train
`funnybirds-mcbm-rl-g*` → `CONFIG_PREFIX=funnybirds-mcbm-rl sbatch train/renderer_swap.slurm`.*

In [ ]:
import os, json, re, glob
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
CURATED=Path(os.environ["CURATED_DATA"]); REPO=Path.cwd().parent
import sys; sys.path.insert(0,str(REPO/"analysis"))
try:
    from plotting import set_paper_style, PALETTE; set_paper_style(); CBM_C,MCBM_C=PALETTE["CBM"],PALETTE["MCBM"]
except Exception: CBM_C,MCBM_C="#0072B2","#D55E00"
plt.rcParams["figure.dpi"]=120; EPS=1e-3; ORDER=["tail","wing","beak","foot","eye"]; RL_C="#009E73"
def need(p,how):
    ok=Path(p).exists()
    if not ok: print(f"[pending] {p}\n  produce it:  {how}")
    return ok
def load_swaps(prefix):
    rows=[]
    for fp in sorted(glob.glob(str(CURATED/"swap"/f"{prefix}-g*-s1.csv"))):
        m=re.search(r"-g([0-9p]+)-s",Path(fp).name)
        if m: d=pd.read_csv(fp); d["gamma"]=float(m.group(1).replace("p",".")); rows.append(d)
    return pd.concat(rows,ignore_index=True) if rows else None


## 1 · Standard vs relabeled — tail z-ordering vs γ  *(the headline)*
The single comparison the whole notebook is for: does image-level relabeling raise the
tail's swap-detection accuracy above the standard model's?

In [ ]:
STD=load_swaps("funnybirds-mcbm"); RL=load_swaps("funnybirds-mcbm-rl")
if RL is None:
    print("[pending] relabeled swap CSVs -> build relabel labels + train funnybirds-mcbm-rl + "
          "CONFIG_PREFIX=funnybirds-mcbm-rl sbatch train/renderer_swap.slurm")
else:
    b=RL[RL.part=="tail"].groupby("gamma").ordering_correct.mean()
    fig,ax=plt.subplots(figsize=(6,3.6)); x=np.arange(len(b))
    ax.plot(x,b.values,"s-",color=RL_C,label="relabeled (image-level)")
    if STD is not None:
        a=STD[STD.part=="tail"].groupby("gamma").ordering_correct.mean().reindex(b.index)
        ax.plot(x,a.values,"o-",color=MCBM_C,label="standard (species-level)")
        display(pd.DataFrame({"standard":a,"relabeled":b}).round(3))
    ax.axhline(0.5,ls=":",color="gray"); ax.axhline(1.0,ls=":",color="green")
    ax.set_xticks(x); ax.set_xticklabels([f"γ={g:g}" for g in b.index]); ax.set_ylim(0,1.05)
    ax.set_ylabel("tail ordering_correct"); ax.legend(); ax.set_title("Does image-level relabeling fix tail grounding?")

📊 **The causal verdict.** relabeled (green) ≫ standard (orange) ⇒ the species-level labels *caused* the backwash — your §3b critique proven causally. relabeled ≈ standard ⇒ backwash is intrinsic to the bottleneck (a deeper problem than labels).

## 2 · Relabeled grounding heatmap (part × γ)
The full per-part view for the relabeled model — where does relabeling help, where not?

In [ ]:
if RL is not None:
    H=RL.groupby(["gamma","part"]).ordering_correct.mean().unstack().reindex(columns=ORDER)
    display(H.round(3))
    fig,ax=plt.subplots(figsize=(6.2,3.8)); im=ax.imshow(H.values,cmap="RdYlGn",vmin=0,vmax=1,aspect="auto")
    ax.set_xticks(range(len(ORDER))); ax.set_xticklabels(ORDER); ax.set_yticks(range(len(H.index)))
    ax.set_yticklabels([f"γ={g:g}" for g in H.index])
    for i in range(H.shape[0]):
        for j in range(H.shape[1]):
            v=H.values[i,j]
            if np.isfinite(v): ax.text(j,i,f"{v:.2f}",ha="center",va="center",fontsize=8)
    ax.set_title("Relabeled MCBM grounding: part × γ"); fig.colorbar(im,fraction=0.046)
else: print("[pending] relabeled swap CSVs")

📊 Compare this grid to notebook 03 §6 (standard). Greener tail column here = relabeling grounded the tail.

## 3 · Deletion backwash — standard vs relabeled
The renderer-free cross-check: does the relabeled model retain a deleted part's concept
less than the standard model? Reads the grounding parquets.

In [ ]:
def tail_ret(prefix):
    fs=sorted(glob.glob(str(CURATED/"grounding"/f"{prefix}-g*-s1.parquet")))
    out={}
    for fp in fs:
        m=re.search(r"-g([0-9p]+)-s",Path(fp).name); 
        if not m: continue
        d=pd.read_parquet(fp); d=d[d.part=="tail"]
        if "changed_frac" in d.columns: d=d[d.changed_frac>EPS]
        if len(d) and d.p_intact.mean()>1e-6: out[float(m.group(1).replace("p","."))]=d.p_removed.mean()/d.p_intact.mean()
    return pd.Series(out).sort_index()
a=tail_ret("funnybirds-mcbm"); b=tail_ret("funnybirds-mcbm-rl")
if len(b):
    D=pd.DataFrame({"standard":a,"relabeled":b}); display(D.round(3))
    fig,ax=plt.subplots(figsize=(6,3.4)); x=np.arange(len(D))
    ax.plot(x,D["standard"],"o-",color=MCBM_C,label="standard"); ax.plot(x,D["relabeled"],"s-",color=RL_C,label="relabeled")
    ax.set_xticks(x); ax.set_xticklabels([f"γ={g:g}" for g in D.index]); ax.set_ylim(0,1.02)
    ax.set_ylabel("tail retained_frac (visible-only)"); ax.legend(); ax.set_title("Deletion backwash: standard vs relabeled")
else: print("[pending] relabeled grounding parquets -> run grounding on funnybirds-mcbm-rl models")

📊 Lower relabeled retention corroborates the swap result: the label fix reduces backwash on an independent probe.

## Takeaway
This notebook isolates **cause**. If relabeling grounds the tail (swap ↑, deletion ↓),
the backwash in CBM/MCBM is an artifact of **species-level label standardization** — fix
the labels, fix the leakage. If it doesn't, the bottleneck itself is the problem. Either
way it converts the correlational §3b critique into a controlled experiment.